# FMCW range-Doppler maps: the phone app

Companion notebook to the Page. Every module runs here, in your browser, with no Google account and no server: FFT basics, range-Doppler on a simulated cube, the 2D FFT, windowing, RaDICaL real data, MTI, CA-CFAR, and the angle FFT.

**How to use it:**
- **Run → Run All Cells** runs everything top to bottom (about a minute after the first load). Or tap a cell and press ▶ in the toolbar; the setup and helper cells have to run first.
- Python runs on the phone itself (Pyodide). The first open downloads Python and numpy/scipy/matplotlib (roughly 30–40 MB); later opens come from Safari's cache.
- The RaDICaL data is already bundled: 9 frames of `indoor_sample_50.h5`, stored losslessly in `radical_frames.npz` (2.2 MB).
- Your edits save in this browser automatically (File → Save also works). To start over, use File → Reload Notebook from Disk, or delete the notebook from the file browser; the original comes back on the next open.
- Unfolded, plots fill the width; pinch to zoom for detail.

In [ ]:
#@title Setup (imports, plots sized for ~800 px)
import sys, os, io, math, time
if sys.platform == "emscripten":          # in the browser (Pyodide)
    import piplite
    await piplite.install(["numpy", "scipy", "matplotlib"])
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal.windows import hann
%matplotlib inline
plt.rcParams.update({"figure.figsize": (7.6, 4.2), "figure.dpi": 100,
                     "font.size": 10, "axes.titlesize": 11})
HELPERS = "/tmp/fmcw_helpers"
os.makedirs(HELPERS, exist_ok=True)
if HELPERS not in sys.path:
    sys.path.insert(0, HELPERS)
import scipy, matplotlib
print("numpy", np.__version__, "| scipy", scipy.__version__, "| matplotlib", matplotlib.__version__)

### Helper modules
The next four cells write the project's scripts (`fft2d_explainer.py`, `windowing_compare.py`, `ca_cfar_2d.py`, `angle_fft.py`) into the browser's Python so later cells can import them. Their code is collapsed; tap the bar to the left of a cell to expand it. Run all does this for you.

In [ ]:
# Helper: write `fft2d_explainer.py` (tap ▶, no need to read)
SRC = r'''#!/usr/bin/env python3
"""Show that a range FFT then a Doppler FFT is just a 2-D FFT.

On the Sunday simulation cube, an FFT along fast time (axis 0) followed
by an FFT along slow time (axis 1) matches np.fft.fft2 to numerical
roundoff.  The two axes commute, and the cost is N M log(N M).  The
script also draws a picture-domain example (stripes and a rectangle)
so the mapping from orientation and frequency onto the 2-D spectrum
is visible, and explains why we fftshift only the Doppler axis.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


C, FC = 3.0e8, 77e9
B, TC, T = 150e6, 40e-6, 50e-6
NS, NC = 256, 128
S, FS, LAM = B / TC, NS / TC, C / FC
TARGETS = [(50.0, 0.0, 1.0), (80.0, 8.0, 0.85), (120.0, -12.0, 0.7)]


def make_sim_cube(noise_std: float = 0.15, seed: int = 7) -> np.ndarray:
    """Build the Sunday IF cube: samples x chirps, three point targets."""
    m, n = np.arange(NS), np.arange(NC)
    cube = np.zeros((NS, NC), dtype=np.complex128)
    for R, v, A in TARGETS:
        fb = 2.0 * S * R / C
        cube += (
            A
            * np.exp(1j * 2.0 * np.pi * fb * m / FS)[:, None]
            * np.exp(1j * 4.0 * np.pi * v * n * T / LAM)[None, :]
        )
    rng = np.random.default_rng(seed)
    cube += noise_std * (
        rng.normal(size=cube.shape) + 1j * rng.normal(size=cube.shape)
    ) / np.sqrt(2.0)
    return cube


def separable_ffts(cube: np.ndarray) -> np.ndarray:
    """Range FFT along axis 0, then Doppler FFT along axis 1."""
    rd = np.fft.fft(cube, axis=0)
    rd = np.fft.fft(rd, axis=1)
    return rd


def compare_to_fft2(cube: np.ndarray) -> dict:
    """Return max-abs errors for (range then Doppler), the swapped order, and fft2."""
    rd_range_then_doppler = separable_ffts(cube)
    rd_doppler_then_range = np.fft.fft(np.fft.fft(cube, axis=1), axis=0)
    rd_fft2 = np.fft.fft2(cube)
    # Separable Hann windows: two 1-D tapers vs one outer-product taper.
    from scipy.signal.windows import hann

    wr = hann(cube.shape[0], sym=False)
    wd = hann(cube.shape[1], sym=False)
    win = wr[:, None] * wd[None, :]
    rd_win_sep = np.fft.fft(cube * wr[:, None], axis=0)
    rd_win_sep = np.fft.fft(rd_win_sep * wd[None, :], axis=1)
    rd_win_2d = np.fft.fft2(cube * win)
    return {
        "max_abs_range_then_doppler": float(np.max(np.abs(rd_range_then_doppler - rd_fft2))),
        "max_abs_doppler_then_range": float(np.max(np.abs(rd_doppler_then_range - rd_fft2))),
        "max_abs_windowed": float(np.max(np.abs(rd_win_sep - rd_win_2d))),
        "shape": cube.shape,
        "cost_1d_naive": int(cube.size * (cube.shape[0] + cube.shape[1])),
        "n": int(cube.shape[0]),
        "m": int(cube.shape[1]),
        "nm_log_nm": float(cube.size * np.log2(cube.size)),
    }


def fftshift_demo(n_doppler: int = 8) -> dict:
    """Bin order of an n-point FFT before and after fftshift."""
    raw = np.fft.fftfreq(n_doppler, d=1.0) * n_doppler  # bins: 0,1,...,n/2-1,-n/2,...
    shifted = np.fft.fftshift(raw)
    return {"n": n_doppler, "before": raw.tolist(), "after": shifted.tolist()}


def make_example_images(n: int = 128) -> dict:
    """Vertical stripes, 45-degree stripes, and a filled rectangle."""
    x = np.arange(n)
    xx, yy = np.meshgrid(x, x)
    vertical = np.cos(2.0 * np.pi * 8.0 * xx / n)
    diagonal = np.cos(2.0 * np.pi * (6.0 * xx + 6.0 * yy) / n)
    rectangle = np.zeros((n, n), dtype=np.float64)
    rectangle[48:80, 36:92] = 1.0  # taller-than-wide? 32 rows x 56 cols — wider in x
    return {"vertical": vertical, "diagonal": diagonal, "rectangle": rectangle}


def spectrum_db(img: np.ndarray) -> np.ndarray:
    spec = np.fft.fftshift(np.fft.fft2(img))
    power = np.abs(spec) ** 2
    db = 10.0 * np.log10(power + 1e-18)
    return db - db.max()


def make_plot(images: dict, stats: dict, out_path: Path) -> None:
    labels = [
        (
            "vertical",
            "Vertical stripes\n(oscillates along x)",
            "Spectrum: energy on the\nhorizontal frequency axis",
        ),
        (
            "diagonal",
            "Diagonal stripes\n(equal fx and fy)",
            "Spectrum: peaks sit on the\nperpendicular diagonal",
        ),
        (
            "rectangle",
            "Filled rectangle\n(wide in x, shorter in y)",
            "Spectrum: 2-D sinc, narrower\nwhere the shape was wider",
        ),
    ]
    fig, axes = plt.subplots(2, 3, figsize=(12.4, 7.6), constrained_layout=True)
    fig.suptitle(
        "A 2-D FFT is two 1-D FFTs: orientation and frequency map across\n"
        f"On the radar cube, fft(axis=0) then fft(axis=1) vs fft2: "
        f"max |Δ| = {stats['max_abs_range_then_doppler']:.3e}"
    )
    for col, (key, title, spec_title) in enumerate(labels):
        img = images[key]
        axes[0, col].imshow(img, cmap="gray", origin="upper", vmin=-1 if key != "rectangle" else 0, vmax=1)
        axes[0, col].set_title(title, fontsize=10)
        axes[0, col].set_xlabel("x (horizontal)")
        axes[0, col].set_ylabel("y (vertical)")
        axes[0, col].set_xticks([])
        axes[0, col].set_yticks([])

        db = spectrum_db(img)
        im = axes[1, col].imshow(
            db, cmap="viridis", origin="lower", vmin=-50, vmax=0
        )
        axes[1, col].set_title(spec_title, fontsize=10)
        axes[1, col].set_xlabel("fx (after fftshift)")
        axes[1, col].set_ylabel("fy (after fftshift)")
        axes[1, col].set_xticks([])
        axes[1, col].set_yticks([])
        fig.colorbar(im, ax=axes[1, col], fraction=0.046, pad=0.04, label="dB")

    fig.savefig(out_path, dpi=140)
    plt.close(fig)


def pool_for_heatmap(db: np.ndarray, n: int = 16) -> np.ndarray:
    """Max-pool a square spectrum into an n x n grid, clipped to 0..40 dB above -40."""
    h, w = db.shape
    rs, cs = h // n * n, w // n * n
    sl = db[:rs, :cs].reshape(n, rs // n, n, cs // n).max(axis=(1, 3))
    return np.clip(sl + 40.0, 0.0, 40.0)


def main() -> int:
    cube = make_sim_cube()
    stats = compare_to_fft2(cube)
    shift = fftshift_demo(8)
    print("=== 2-D FFT identity on the Sunday sim cube ===")
    print(f"cube shape (fast time x slow time): {stats['shape']}")
    print(f"max |fft(axis=0) then fft(axis=1) - fft2| = {stats['max_abs_range_then_doppler']:.6e}")
    print(f"max |fft(axis=1) then fft(axis=0) - fft2| = {stats['max_abs_doppler_then_range']:.6e}")
    print(f"max |windowed separable - windowed fft2|   = {stats['max_abs_windowed']:.6e}")
    n, m = stats["n"], stats["m"]
    print(
        f"complexity: M * O(N log N) + N * O(M log M) "
        f"= O(NM log(NM)) = O({n}*{m}*log2({n*m})) ~ {stats['nm_log_nm']:.1f} ops"
    )
    print("=== fftshift Doppler demo (Nc = 8, units of bins) ===")
    print("before:", ", ".join(f"{v:+.0f}" for v in shift["before"]))
    print("after: ", ", ".join(f"{v:+.0f}" for v in shift["after"]))

    images = make_example_images(128)
    out = Path("/workspace/fft2d_explainer.png")
    make_plot(images, stats, out)
    print(f"saved {out} ({out.stat().st_size} bytes)")

    rect_spec = spectrum_db(images["rectangle"])
    pooled = pool_for_heatmap(rect_spec, 16)
    np.set_printoptions(precision=1, linewidth=140, suppress=True)
    print("rectangle spectrum pooled 16x16 (0..40 = dB above -40):")
    print(np.round(pooled, 1))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''
open(os.path.join(HELPERS, 'fft2d_explainer.py'), 'w').write(SRC)
print('wrote fft2d_explainer.py', len(SRC), 'chars')

In [ ]:
# Helper: write `windowing_compare.py` (tap ▶, no need to read)
SRC = r'''#!/usr/bin/env python3
"""Compare rectangular, Hann, Blackman, and Chebyshev (80 dB) windows.

Each window is applied on both range (fast time) and Doppler (slow time).
Metrics are measured from the highly zero-padded DTFT of the window itself
(peak sidelobe level, -3 dB mainlobe width in bins, processing / SNR loss)
and cross-checked on range cuts of:

  * the Sunday three-target simulation cube (cut through the 50 m target)
  * RaDICaL indoor_sample_50.h5 frame 30 (zero-Doppler cut through the wall)

A separate masking cube places a -40 dB companion three range bins from a
slightly off-grid strong target, to show how rectangular sidelobes hide it
while Blackman (and Chebyshev) let it poke through.
"""

from __future__ import annotations

from pathlib import Path

try:
    import h5py  # only the command-line main() needs it
except ImportError:
    h5py = None
import numpy as np
from scipy.signal.windows import blackman, chebwin, hann


# Sunday simulation (matches the Page skeleton).
C, FC = 3.0e8, 77e9
B, TC, T = 150e6, 40e-6, 50e-6
NS, NC = 256, 128
S, FS, LAM = B / TC, NS / TC, C / FC
RANGE_RES = C / (2.0 * B)  # 1.00 m
TARGETS = [(50.0, 0.0, 1.0), (80.0, 8.0, 0.85), (120.0, -12.0, 0.7)]

# RaDICaL indoor_human_rcs.cfg (same constants as the rest of this project).
RAD_SLOPE, RAD_FS = 100e12, 9499e3
RAD_NS, RAD_NC, RAD_NTX = 304, 32, 2
RAD_T = RAD_NTX * (58e-6 + 40e-6)
RAD_B = RAD_SLOPE * (RAD_NS / RAD_FS)
RAD_DR = C / (2.0 * RAD_B)


def make_window(name: str, n: int) -> np.ndarray:
    if name == "rectangular":
        return np.ones(n, dtype=np.float64)
    if name == "hann":
        return hann(n, sym=False).astype(np.float64)
    if name == "blackman":
        return blackman(n, sym=False).astype(np.float64)
    if name == "chebyshev-80dB":
        return chebwin(n, at=80.0, sym=False).astype(np.float64)
    raise KeyError(name)


WINDOW_NAMES = ("rectangular", "hann", "blackman", "chebyshev-80dB")


def processing_loss_db(w: np.ndarray) -> float:
    """SNR loss vs a rectangular window of the same length, in dB.

    PL = -10 log10( |sum w|^2 / (N sum w^2) ).  Zero for all-ones.
    Equivalently 10 log10(ENBW) where ENBW = N sum(w^2) / |sum w|^2.
    """
    w = np.asarray(w, dtype=np.float64)
    n = w.size
    return float(-10.0 * np.log10(np.abs(w.sum()) ** 2 / (n * np.sum(w * w))))


def _interp_cross(bins: np.ndarray, db: np.ndarray, i: int, step: int, level: float) -> float:
    """Linear interpolation of the bin where `db` crosses `level` from i toward step."""
    n = db.size
    while 0 <= i + step < n and db[i + step] > level:
        i += step
    j = i + step
    if j < 0 or j >= n:
        return float(bins[i])
    t = (level - db[i]) / (db[j] - db[i] + 1e-30)
    return float(bins[i] + t * (bins[j] - bins[i]))


def first_null_indices(db: np.ndarray, peak: int) -> tuple[int, int]:
    """Walk outward from the peak while the response is still falling."""

    def walk(start: int, step: int) -> int:
        i = start
        n = db.size
        while 0 <= i + step < n and db[i + step] <= db[i] + 1e-9:
            i += step
        return i

    return walk(peak, -1), walk(peak, +1)


def window_metrics(w: np.ndarray, pad: int = 64) -> dict:
    """PSL, -3 dB width (bins of the original N-point DFT), and processing loss."""
    w = np.asarray(w, dtype=np.float64)
    n = w.size
    nfft = pad * n
    spec = np.fft.fftshift(np.fft.fft(w, n=nfft))
    mag = np.abs(spec)
    mag = mag / mag.max()
    db = 20.0 * np.log10(mag + 1e-20)
    bins = (np.arange(nfft) - nfft // 2) * (n / nfft)
    peak = int(np.argmax(db))
    left, right = first_null_indices(db, peak)
    outside = np.ones(db.size, dtype=bool)
    outside[left : right + 1] = False
    psl = float(db[outside].max()) if outside.any() else float("nan")
    width = _interp_cross(bins, db, peak, 1, -3.0) - _interp_cross(bins, db, peak, -1, -3.0)
    return {
        "n": n,
        "psl_db": psl,
        "width_3db_bins": float(width),
        "processing_loss_db": processing_loss_db(w),
        "bins": bins,
        "db": db,
    }


def make_sim_cube(
    targets=TARGETS,
    noise_std: float = 0.15,
    seed: int = 7,
) -> np.ndarray:
    m, n = np.arange(NS), np.arange(NC)
    cube = np.zeros((NS, NC), dtype=np.complex128)
    for R, v, A in targets:
        fb = 2.0 * S * R / C
        cube += (
            A
            * np.exp(1j * 2.0 * np.pi * fb * m / FS)[:, None]
            * np.exp(1j * 4.0 * np.pi * v * n * T / LAM)[None, :]
        )
    if noise_std:
        rng = np.random.default_rng(seed)
        cube += noise_std * (
            rng.normal(size=cube.shape) + 1j * rng.normal(size=cube.shape)
        ) / np.sqrt(2.0)
    return cube


def range_cut_sim(cube: np.ndarray, wr: np.ndarray, wd: np.ndarray, pad: int = 16) -> tuple[np.ndarray, np.ndarray]:
    """Zero-padded range profile at zero Doppler, linear power."""
    ns, nc = cube.shape
    x = cube * wr[:, None]
    rng = np.fft.fft(x, n=pad * ns, axis=0)
    rng = rng * wd[None, :]
    rd = np.fft.fftshift(np.fft.fft(rng, axis=1), axes=1)
    # Zero Doppler sits at the centre after fftshift.
    power = np.abs(rd[:, nc // 2]) ** 2
    r_axis = np.arange(pad * ns) * RANGE_RES / pad
    return r_axis, power


def metrics_from_cut(
    r_axis: np.ndarray,
    power: np.ndarray,
    r0: float,
    r_window: float = 18.0,
    bin_m: float | None = None,
) -> dict:
    """Measure PSL and -3 dB width around the peak nearest r0."""
    sl = (r_axis >= r0 - r_window) & (r_axis <= r0 + r_window)
    idx = np.flatnonzero(sl)
    p = power[idx]
    r = r_axis[idx]
    db = 10.0 * np.log10(p + 1e-30)
    db = db - db.max()
    # Convert range to bin units of the original (unpadded) DFT.
    metres_per_bin = RANGE_RES if bin_m is None else bin_m
    bins = (r - r[np.argmax(db)]) / metres_per_bin
    peak = int(np.argmax(db))
    left, right = first_null_indices(db, peak)
    outside = np.ones(db.size, dtype=bool)
    outside[left : right + 1] = False
    psl = float(db[outside].max()) if outside.any() else float("nan")
    # Reuse interpolator; build a local bin axis centred on the peak.
    width = _interp_cross(bins, db, peak, 1, -3.0) - _interp_cross(bins, db, peak, -1, -3.0)
    return {
        "psl_db": psl,
        "width_3db_bins": float(width),
        "peak_range_m": float(r[peak]),
        "peak_db": float(10.0 * np.log10(p.max() + 1e-30)),
    }


def range_cut_radical(
    cube: np.ndarray, wr: np.ndarray, wd: np.ndarray, pad: int = 8
) -> tuple[np.ndarray, np.ndarray]:
    """Zero-Doppler range profile, power averaged over the 8 virtual RX."""
    x = np.asarray(cube, dtype=np.complex128)
    ns = x.shape[-1]
    x = x * wr[None, None, :]
    x = np.fft.fft(x, n=pad * ns, axis=-1)
    x = x * wd[:, None, None]
    x = np.fft.fftshift(np.fft.fft(x, axis=0), axes=0)
    power = np.mean(np.abs(x) ** 2, axis=1)  # (n_doppler, n_range)
    n_d = power.shape[0]
    cut = power[n_d // 2]
    r_axis = np.arange(pad * ns) * RAD_DR / pad
    return r_axis, cut


def make_plot(
    win_curves: dict,
    sim_cuts: dict,
    mask_cuts: dict,
    rad_cuts: dict,
    out_path: Path,
) -> None:
    import matplotlib.pyplot as plt

    colors = {
        "rectangular": "C3",
        "hann": "C0",
        "blackman": "C2",
        "chebyshev-80dB": "C1",
    }
    fig, axes = plt.subplots(2, 2, figsize=(12.6, 8.4), constrained_layout=True)
    fig.suptitle("Windowing on range and Doppler: sidelobes, width, and a hidden companion")

    ax = axes[0, 0]
    for name, met in win_curves.items():
        ax.plot(met["bins"], met["db"], color=colors[name], lw=1.4, label=name)
    ax.set_xlim(-8, 8)
    ax.set_ylim(-90, 3)
    ax.set_xlabel("Frequency (bins of the original DFT)")
    ax.set_ylabel("Magnitude (dB)")
    ax.set_title("Window DTFT (N = 256, 64x padded)")
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.grid(True, alpha=0.3)
    ax.axhline(-3, color="0.5", ls=":", lw=0.8)

    ax = axes[0, 1]
    for name, (r, p) in sim_cuts.items():
        db = 10.0 * np.log10(p + 1e-30)
        db = db - db.max()
        ax.plot(r, db, color=colors[name], lw=1.3, label=name)
    ax.set_xlim(40, 70)
    ax.set_ylim(-80, 3)
    ax.set_xlabel("Range (m)")
    ax.set_ylabel("Power (dB, peak-normalized)")
    ax.set_title("Sim range cut through the 50 m target (zero Doppler)")
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.grid(True, alpha=0.3)

    ax = axes[1, 0]
    for name in ("rectangular", "hann", "blackman", "chebyshev-80dB"):
        r, p = mask_cuts[name]
        db = 10.0 * np.log10(p + 1e-30)
        # Normalize to the strong-target peak (near 50.25 m), not a far sidelobe.
        near = (r >= 47) & (r <= 52)
        db = db - db[near].max()
        ax.plot(r, db, color=colors[name], lw=1.3, label=name)
    ax.axvline(50.25, color="0.35", ls="--", lw=0.9)
    ax.axvline(53.25, color="0.35", ls="--", lw=0.9)
    ax.axhline(-40, color="0.45", ls=":", lw=0.8)
    ax.text(50.25, -7, "strong", ha="center", fontsize=8, color="0.3")
    ax.text(53.25, -36, "-40 dB, +3 bins", ha="left", fontsize=8, color="0.3")
    ax.set_xlim(46, 62)
    ax.set_ylim(-80, 3)
    ax.set_xlabel("Range (m)")
    ax.set_ylabel("Power (dB, strong-peak-normalized)")
    ax.set_title("Masking: -40 dB companion 3 bins from a strong target")
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    for name, (r, p) in rad_cuts.items():
        db = 10.0 * np.log10(p + 1e-30)
        db = db - db.max()
        ax.plot(r, db, color=colors[name], lw=1.2, label=name)
    ax.set_xlim(0, 12)
    ax.set_ylim(-50, 3)
    ax.set_xlabel("Range (m)")
    ax.set_ylabel("Power (dB, peak-normalized)")
    ax.set_title("RaDICaL frame 30, raw zero-Doppler range cut")
    ax.legend(loc="upper right", fontsize=8, frameon=False)
    ax.grid(True, alpha=0.3)

    fig.savefig(out_path, dpi=140)
    plt.close(fig)


def main() -> int:
    print("=== Window DTFT metrics (periodic windows, N = 256) ===")
    win_curves = {}
    rows = []
    for name in WINDOW_NAMES:
        w = make_window(name, NS)
        met = window_metrics(w)
        win_curves[name] = met
        # Same window family on Doppler (N = 128) — loss depends weakly on N.
        wd = make_window(name, NC)
        pl_d = processing_loss_db(wd)
        rows.append(
            {
                "name": name,
                "psl_db": met["psl_db"],
                "width_3db_bins": met["width_3db_bins"],
                "pl_range_db": met["processing_loss_db"],
                "pl_doppler_db": pl_d,
                "pl_2d_db": met["processing_loss_db"] + pl_d,
            }
        )
        print(
            f"{name:18s}  PSL {met['psl_db']:7.2f} dB   "
            f"-3 dB width {met['width_3db_bins']:5.3f} bins   "
            f"PL {met['processing_loss_db']:5.2f} dB (range N=256) / "
            f"{pl_d:5.2f} dB (Doppler N=128) / "
            f"2-D {met['processing_loss_db'] + pl_d:5.2f} dB"
        )

    print("\n=== Same metrics at RaDICaL lengths (Ns = 304, Nc = 32) ===")
    for name in WINDOW_NAMES:
        mr = window_metrics(make_window(name, RAD_NS))
        md_pl = processing_loss_db(make_window(name, RAD_NC))
        print(
            f"{name:18s}  PSL {mr['psl_db']:7.2f} dB   "
            f"-3 dB width {mr['width_3db_bins']:5.3f} bins   "
            f"PL {mr['processing_loss_db']:5.2f} dB (range) / {md_pl:5.2f} dB (Doppler)"
        )

    cube = make_sim_cube()
    print("\n=== Empirical range-cut metrics, sim 50 m target (both axes windowed) ===")
    sim_cuts = {}
    for name in WINDOW_NAMES:
        wr, wd = make_window(name, NS), make_window(name, NC)
        r, p = range_cut_sim(cube, wr, wd)
        sim_cuts[name] = (r, p)
        emp = metrics_from_cut(r, p, 50.0, r_window=18.0)
        print(
            f"{name:18s}  peak {emp['peak_range_m']:5.2f} m   "
            f"PSL {emp['psl_db']:7.2f} dB   "
            f"-3 dB width {emp['width_3db_bins']:5.3f} bins"
        )

    # Masking cube: strong slightly off-grid so rectangular sidelobes exist
    # at neighboring bins; weak companion exactly 3 range bins later, -40 dB.
    # Tiny noise so the floor sits well below -40 dB.
    mask_targets = [(50.25, 0.0, 1.0), (53.25, 0.0, 10 ** (-40.0 / 20.0))]
    mask_cube = make_sim_cube(targets=mask_targets, noise_std=3e-4, seed=1)
    print("\n=== Masking cube: strong 50.25 m + weak 53.25 m at -40 dB ===")
    mask_cuts = {}
    for name in WINDOW_NAMES:
        wr, wd = make_window(name, NS), make_window(name, NC)
        r, p = range_cut_sim(mask_cube, wr, wd)
        mask_cuts[name] = (r, p)
        # Level at 53.25 m relative to the strong peak.
        i_s = int(np.argmin(np.abs(r - 50.25)))
        i_w = int(np.argmin(np.abs(r - 53.25)))
        # Local maximum within ±0.4 m of 53.25, vs strong peak.
        near = (r >= 52.85) & (r <= 53.65)
        level = 10.0 * np.log10((p[near].max() + 1e-30) / (p[i_s] + 1e-30))
        print(
            f"{name:18s}  local max near 53.25 m: {level:6.2f} dB "
            f"(sample at 53.25 m: {10*np.log10((p[i_w]+1e-30)/(p[i_s]+1e-30)):6.2f} dB)"
        )

    print("\n=== RaDICaL frame 30, raw zero-Doppler cut (wall near 9.2 m) ===")
    with h5py.File("/workspace/radar_recordings/indoor_sample_50.h5", "r") as f:
        rad = np.asarray(f["radar"][30])
    print(f"loaded cube {rad.shape}")
    rad_cuts = {}
    for name in WINDOW_NAMES:
        wr, wd = make_window(name, RAD_NS), make_window(name, RAD_NC)
        r, p = range_cut_radical(rad, wr, wd)
        rad_cuts[name] = (r, p)
        emp = metrics_from_cut(r, p, 9.2, r_window=1.6, bin_m=RAD_DR)
        print(
            f"{name:18s}  peak {emp['peak_range_m']:5.2f} m   "
            f"local PSL {emp['psl_db']:7.2f} dB   "
            f"-3 dB width {emp['width_3db_bins']:5.3f} bins"
        )
        # Also report the coupling peak near 0 m for context.
        emp0 = metrics_from_cut(r, p, 0.15, r_window=1.2, bin_m=RAD_DR)
        print(
            f"{'':18s}  coupling peak {emp0['peak_range_m']:5.2f} m   "
            f"-3 dB width {emp0['width_3db_bins']:5.3f} bins"
        )

    out = Path("/workspace/windowing_compare.png")
    make_plot(win_curves, sim_cuts, mask_cuts, rad_cuts, out)
    print(f"\nsaved {out} ({out.stat().st_size} bytes)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''
open(os.path.join(HELPERS, 'windowing_compare.py'), 'w').write(SRC)
print('wrote windowing_compare.py', len(SRC), 'chars')

In [ ]:
# Helper: write `ca_cfar_2d.py` (tap ▶, no need to read)
SRC = r'''#!/usr/bin/env python3
"""2-D cell-averaging CFAR on a range-Doppler power map.

The detector runs in the *linear* power domain (not dB).  Around each cell
under test (CUT) it estimates the local noise as the mean of a ring of
training cells, skipping a guard region so the target's own energy does not
bias the estimate.  The threshold is

    threshold = alpha * mean(training)
    alpha    = N * (P_fa ** (-1/N) - 1)

which is the exact scale that yields false-alarm probability P_fa when the
noise power is i.i.d. exponential (the distribution of |complex Gaussian|^2).
N is the number of training cells that were actually available after edge
clipping, so cells near the border use a slightly different alpha rather
than being silently skipped or wrapped.
"""

from __future__ import annotations

import numpy as np
from scipy.ndimage import label, maximum_filter, uniform_filter


def cfar_scale(n_train: np.ndarray | float, pfa: float) -> np.ndarray | float:
    """Return alpha(N, P_fa) = N * (P_fa^(-1/N) - 1).

    Works on scalars or arrays.  N <= 0 is mapped to +inf so those cells
    can never declare a detection.
    """
    n = np.asarray(n_train, dtype=np.float64)
    alpha = np.full_like(n, np.inf, dtype=np.float64)
    ok = n > 0
    alpha[ok] = n[ok] * (np.power(pfa, -1.0 / n[ok]) - 1.0)
    return alpha if alpha.ndim else float(alpha)


def _rect_sum(img: np.ndarray, half_r: int, half_d: int) -> np.ndarray:
    """Sum of a (2*half_r+1) by (2*half_d+1) window at every pixel.

    Edges are padded with zeros (no wrap-around), so the sum near a border
    only includes the cells that actually exist.
    """
    size = (2 * half_r + 1, 2 * half_d + 1)
    # uniform_filter returns the mean; multiply back by the window area
    # (including padded zeros) to recover the sum of the real pixels.
    return uniform_filter(img.astype(np.float64), size=size, mode="constant", cval=0.0) * (
        size[0] * size[1]
    )


def ca_cfar_2d(
    power: np.ndarray,
    *,
    guard: tuple[int, int] = (3, 1),
    train: tuple[int, int] = (10, 4),
    pfa: float = 1e-5,
    min_train: int | None = None,
) -> dict:
    """2-D CA-CFAR on a linear-power range-Doppler map.

    Parameters
    ----------
    power
        2-D array of *linear* power with shape (n_range, n_doppler).
    guard
        (guard_range, guard_doppler): cells excluded on each side of the
        CUT so a spread target does not leak into the noise estimate.
    train
        (train_range, train_doppler): training cells beyond the guard
        on each side.
    pfa
        Design false-alarm probability per cell, used to set alpha.
    min_train
        Require at least this many training cells (default: half the
        nominal interior count).  Cells with fewer are not tested.

    Returns
    -------
    dict with
        detections : bool array, same shape as ``power``
        noise      : local noise estimate (mean of training cells)
        alpha      : per-cell threshold scale
        threshold  : alpha * noise
        n_train    : per-cell training-cell count
    """
    if power.ndim != 2:
        raise ValueError("power must be 2-D (n_range, n_doppler)")
    guard_r, guard_d = (int(g) for g in guard)
    train_r, train_d = (int(t) for t in train)
    if min(guard_r, guard_d, train_r, train_d) < 0 or min(train_r, train_d) < 1:
        raise ValueError("guard cells must be >= 0 and training cells >= 1")

    # Window half-sizes: training sits *outside* the guard box.
    win_r = train_r + guard_r
    win_d = train_d + guard_d

    ones = np.ones_like(power, dtype=np.float64)
    win_sum = _rect_sum(power, win_r, win_d)
    win_cnt = _rect_sum(ones, win_r, win_d)
    # The guard box includes the CUT itself.
    grd_sum = _rect_sum(power, guard_r, guard_d)
    grd_cnt = _rect_sum(ones, guard_r, guard_d)

    n_train = win_cnt - grd_cnt
    train_sum = win_sum - grd_sum
    # Avoid divide-by-zero; unused cells are masked out below.
    noise = np.divide(train_sum, n_train, out=np.zeros_like(train_sum), where=n_train > 0)

    nominal_n = (2 * win_r + 1) * (2 * win_d + 1) - (2 * guard_r + 1) * (2 * guard_d + 1)
    if min_train is None:
        min_train = max(8, nominal_n // 2)

    alpha = cfar_scale(n_train, pfa)
    threshold = alpha * noise
    detections = (power > threshold) & (n_train >= min_train)

    return {
        "detections": detections,
        "noise": noise,
        "alpha": alpha,
        "threshold": threshold,
        "n_train": n_train,
        "nominal_n": nominal_n,
        "min_train": min_train,
    }


def group_peaks(
    power: np.ndarray,
    detections: np.ndarray,
    neighborhood: tuple[int, int] = (3, 3),
) -> list[tuple[int, int]]:
    """Collapse CFAR hits into one peak per connected cluster.

    1. Label 8-connected components of the detection mask.
    2. Inside each component, keep the cell with the largest power
       (ties broken by first occurrence).
    3. As a safety check, drop a peak that is not a local maximum in
       ``neighborhood`` — this rejects plateau fragments.

    Returns a list of (range_index, doppler_index) pairs, sorted by
    descending power.
    """
    if not detections.any():
        return []

    labeled, n_comp = label(detections)
    peaks: list[tuple[int, int]] = []
    local_max = power >= maximum_filter(power, size=neighborhood)
    for comp in range(1, n_comp + 1):
        mask = labeled == comp
        # argmax over the component only
        masked = np.where(mask, power, -np.inf)
        idx = np.unravel_index(int(np.argmax(masked)), power.shape)
        if local_max[idx]:
            peaks.append((int(idx[0]), int(idx[1])))
        else:
            # Still report it: the component max is the right location
            # even if a neighbour just outside the mask is slightly brighter.
            peaks.append((int(idx[0]), int(idx[1])))

    peaks.sort(key=lambda ij: float(power[ij]), reverse=True)
    return peaks


def merge_nearby(
    detections: list[dict],
    *,
    range_tol: float = 1.2,
    vel_tol: float = 0.7,
) -> list[dict]:
    """Merge clustered peaks that are close in range and velocity.

    Detections are assumed sorted by descending power (as ``group_peaks``
    / ``detect`` return them).  Each kept peak swallows later, weaker
    peaks inside a (range_tol, vel_tol) box — typical of a Hann-widened
    target that produced more than one local maximum.
    """
    kept: list[dict] = []
    used = [False] * len(detections)
    for i, det in enumerate(detections):
        if used[i]:
            continue
        used[i] = True
        n_cells = 1
        for j, other in enumerate(detections):
            if used[j]:
                continue
            if (
                abs(other["range_m"] - det["range_m"]) <= range_tol
                and abs(other["velocity_mps"] - det["velocity_mps"]) <= vel_tol
            ):
                used[j] = True
                n_cells += 1
        out = dict(det)
        out["n_merged"] = n_cells
        kept.append(out)
    return kept


def detect(
    power: np.ndarray,
    range_axis: np.ndarray,
    vel_axis: np.ndarray,
    *,
    guard: tuple[int, int] = (3, 1),
    train: tuple[int, int] = (10, 4),
    pfa: float = 1e-5,
    min_range: float = 0.4,
    min_abs_vel: float = 0.0,
    min_snr_db: float = 0.0,
    merge_range: float = 1.2,
    merge_vel: float = 0.7,
) -> tuple[list[dict], dict]:
    """Run CA-CFAR + peak grouping and return physical detections.

    ``min_range`` drops the first few range bins (TX-RX coupling).
    ``min_abs_vel`` optionally ignores leftover near-zero Doppler
    residuals after MTI; use 0 to keep stationary targets (needed for
    the synthetic three-target test).
    ``min_snr_db`` is CUT power over the local CA noise estimate, in dB.
    ``merge_range`` / ``merge_vel`` collapse spread-target fragments;
    set either to 0 to disable merging.
    """
    result = ca_cfar_2d(power, guard=guard, train=train, pfa=pfa)
    det = result["detections"].copy()

    # Drop coupling / invalid range on the mask so those cells never seed a cluster.
    # Velocity and SNR gates are applied *after* grouping: zeroing a Doppler
    # column first can split one spread target into several weaker peaks.
    if min_range > 0:
        det[range_axis < min_range, :] = False

    peaks = group_peaks(power, det)
    detections = []
    for ir, iv in peaks:
        noise = result["noise"][ir, iv] + 1e-30
        detections.append(
            {
                "range_m": float(range_axis[ir]),
                "velocity_mps": float(vel_axis[iv]),
                "range_bin": int(ir),
                "doppler_bin": int(iv),
                "power": float(power[ir, iv]),
                "snr_db": float(10.0 * np.log10(power[ir, iv] / noise)),
            }
        )
    if min_snr_db > 0:
        detections = [d for d in detections if d["snr_db"] >= min_snr_db]
    if min_abs_vel > 0:
        detections = [d for d in detections if abs(d["velocity_mps"]) >= min_abs_vel]
    if merge_range > 0 and merge_vel > 0:
        detections = merge_nearby(detections, range_tol=merge_range, vel_tol=merge_vel)
    result["detections"] = det  # range-gated boolean mask (before clustering)
    return detections, result
'''
open(os.path.join(HELPERS, 'ca_cfar_2d.py'), 'w').write(SRC)
print('wrote ca_cfar_2d.py', len(SRC), 'chars')

In [ ]:
# Helper: write `angle_fft.py` (tap ▶, no need to read)
SRC = r'''#!/usr/bin/env python3
"""Angle FFT on a RaDICaL range-Doppler cube (TDM-MIMO virtual array).

The indoor_human_rcs.cfg used with indoor_sample_50.h5 is an xWR14xx
TDM-MIMO setup:

    channelCfg 15 5 0     # 4 RX (mask 15) and TX0+TX2 (mask 5 = 0b101)
    chirpCfg  0 ... 1     # chirp 0 fires TX0 (hardware TX1)
    chirpCfg  1 ... 4     # chirp 1 fires TX2 (hardware TX3)

On the AWR1443BOOST / IWR1443BOOST the four RX sit on a line at d = λ/2
and the two azimuth transmitters (TX0 and TX2) are 4d = 2λ apart.  The
elevation transmitter (TX1) is unused, so there is no elevation baseline.

Virtual-element positions (in units of d = λ/2), after the SDK reshape
that concatenates TX0's four RX then TX2's four RX, are

    [0, 1, 2, 3, 4, 5, 6, 7] * d

which is a filled 8-element uniform linear array.  The eight H5 channels
are therefore a virtual ULA, not eight simultaneous physical receivers:
they only line up in space after that TX0-then-TX2 ordering, and a moving
target accumulates extra phase during the one-chirp gap between the two
TX slots.  That Doppler-induced phase must be removed before the angle
FFT (TI SWRA554A).  For a target in signed Doppler bin k the correction
on virtual channels belonging to TX index t is

    exp(-j * 2π * k * t / (N_doppler * N_TX))

A far-field source at azimuth θ produces a progressive phase

    Δφ = 2π d sin(θ) / λ

across neighbouring virtual elements.  An N_a-point (zero-padded) FFT of
the windowed snapshot therefore peaks at the bin k satisfying

    sin(θ) = k * λ / (N_a * d) = 2k / N_a     (when d = λ/2)

and the Rayleigh resolution at boresight is θ_res ≈ 2/N radians
(N = 8 → 0.25 rad ≈ 14.3°, matching the visualizer's "15 deg").
"""

from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
from scipy.signal.windows import hann


# Radar constants matching indoor_human_rcs.cfg and the rest of this project.
C = 3.0e8
FC = 77e9
SLOPE = 100e12          # 100 MHz/us
FS = 9499e3             # 9499 ksps
N_SAMPLES = 304
N_CHIRPS = 32           # loops per frame (after TDM de-interleave)
N_TX = 2
N_RX = 4
N_VIRT = N_TX * N_RX    # 8 virtual elements
IDLE_S = 58e-6
RAMP_END_S = 40e-6
T_CHIRP = IDLE_S + RAMP_END_S          # 98 us between TDM slots
T_SLOW = N_TX * T_CHIRP                # 196 us between same-TX chirps
LAM = C / FC                           # 3.896 mm at the 77 GHz start
SPACING = LAM / 2.0                    # designed RX / virtual pitch
B_ADC = SLOPE * (N_SAMPLES / FS)       # bandwidth during the ADC window
RANGE_RES = C / (2.0 * B_ADC)          # 0.0469 m
VEL_RES = LAM / (2.0 * N_CHIRPS * T_SLOW)


def range_axis(n_samples: int = N_SAMPLES) -> np.ndarray:
    return np.arange(n_samples) * RANGE_RES


def vel_axis(n_chirps: int = N_CHIRPS) -> np.ndarray:
    return np.fft.fftshift(np.fft.fftfreq(n_chirps, d=T_SLOW)) * LAM / 2.0


def angle_axis(n_angle: int, spacing: float = SPACING, lam: float = LAM) -> np.ndarray:
    """Azimuth, in radians, for an fftshifted n_angle-point angle FFT."""
    k = np.arange(n_angle) - n_angle // 2
    sin_theta = k * lam / (n_angle * spacing)
    # With d = λ/2 this spans [-1, 1) so every bin is a real angle; clip
    # anyway so a different spacing cannot throw arcsin out of range.
    return np.arcsin(np.clip(sin_theta, -1.0, 1.0))


def range_doppler_cube(cube: np.ndarray, *, mti: bool = False) -> np.ndarray:
    """Range FFT then Doppler FFT, keeping the virtual-array axis.

    Parameters
    ----------
    cube
        Complex IF data shaped (n_chirps, n_virt, n_samples).
    mti
        If True, subtract the slow-time mean on every virtual channel
        before the FFTs (static-clutter suppression).

    Returns
    -------
    rd : ndarray, shape (n_doppler, n_virt, n_range), fftshifted in Doppler.
    """
    x = np.asarray(cube, dtype=np.complex128)
    if mti:
        x = x - x.mean(axis=0, keepdims=True)
    n_chirps, _, n_samples = x.shape
    x = x * hann(n_samples, sym=False)[None, None, :]
    x = np.fft.fft(x, axis=-1)
    x = x * hann(n_chirps, sym=False)[:, None, None]
    x = np.fft.fftshift(np.fft.fft(x, axis=0), axes=0)
    return x


def compensate_tdm_doppler(
    rd: np.ndarray,
    *,
    n_tx: int = N_TX,
    n_rx: int = N_RX,
) -> np.ndarray:
    """Undo the TDM phase walk between transmitters.

    ``rd`` must already be fftshifted along Doppler (axis 0) and laid
    out as [TX0-RX0..RX{n_rx-1}, TX1-RX0.., ...] along axis 1.  Signed
    Doppler bin k (0 at the centre) produces a phase of
    2π k t / (N_doppler N_TX) on every virtual channel of transmitter t;
    multiplying by the conjugate removes it so the 8 channels look like
    a simultaneous ULA.
    """
    n_doppler, n_virt, _ = rd.shape
    if n_virt != n_tx * n_rx:
        raise ValueError(f"expected {n_tx * n_rx} virtual channels, got {n_virt}")
    k = np.arange(n_doppler) - n_doppler // 2
    tx_index = np.repeat(np.arange(n_tx), n_rx)          # [0,0,0,0,1,1,1,1]
    phase = np.exp(
        -1j * 2.0 * np.pi * k[:, None] * tx_index[None, :] / (n_doppler * n_tx)
    )
    return rd * phase[:, :, None]


def angle_fft(
    snapshot: np.ndarray,
    *,
    n_angle: int = 128,
    window: bool = True,
    spacing: float = SPACING,
    lam: float = LAM,
) -> tuple[np.ndarray, np.ndarray]:
    """Windowed, zero-padded angle FFT of one virtual-array snapshot.

    Parameters
    ----------
    snapshot
        Complex samples along the virtual array, shape (n_virt,).
    n_angle
        FFT length.  Values larger than n_virt zero-pad (interpolate)
        the spatial spectrum; they do not invent new resolution.
    window
        Apply a Hann taper along the array (default True).
    spacing, lam
        Element pitch and wavelength used to convert bins to radians.

    Returns
    -------
    spec, theta
        ``spec`` is the complex fftshifted spectrum (length n_angle);
        ``theta`` is the matching azimuth axis in radians.
    """
    x = np.asarray(snapshot, dtype=np.complex128).reshape(-1)
    if window:
        x = x * hann(x.size, sym=False)
    spec = np.fft.fftshift(np.fft.fft(x, n=n_angle))
    return spec, angle_axis(n_angle, spacing=spacing, lam=lam)


def range_angle_map(
    rd: np.ndarray,
    *,
    n_angle: int = 128,
    window: bool = True,
    compensate: bool = True,
) -> tuple[np.ndarray, np.ndarray]:
    """Build a range-azimuth power map from an RD cube.

    Applies optional TDM Doppler compensation, a Hann window and a
    zero-padded angle FFT at every (Doppler, range) cell, then collapses
    Doppler by taking the maximum power.  Raw cubes light up static
    walls; an MTI cube lights up movers.

    Returns
    -------
    power, theta
        ``power`` has shape (n_range, n_angle) in linear units;
        ``theta`` is the azimuth axis in radians.
    """
    work = compensate_tdm_doppler(rd) if compensate else rd
    n_doppler, n_virt, n_range = work.shape
    taper = hann(n_virt, sym=False) if window else np.ones(n_virt)
    # (n_doppler, n_range, n_virt)
    snap = work.transpose(0, 2, 1) * taper
    spec = np.fft.fftshift(np.fft.fft(snap, n=n_angle, axis=-1), axes=-1)
    power = np.max(np.abs(spec) ** 2, axis=0)           # (n_range, n_angle)
    return power, angle_axis(n_angle)


def _quadratic_bin(mag: np.ndarray, peak: int) -> float:
    """Parabolic interpolation of a discrete peak, in bin units."""
    if peak <= 0 or peak >= mag.size - 1:
        return float(peak)
    a, b, c = (float(mag[peak - 1]), float(mag[peak]), float(mag[peak + 1]))
    denom = a - 2.0 * b + c
    if denom == 0.0:
        return float(peak)
    return peak + 0.5 * (a - c) / denom


def estimate_azimuth(
    snapshot: np.ndarray,
    *,
    n_angle: int = 128,
    window: bool = True,
    spacing: float = SPACING,
    lam: float = LAM,
) -> dict:
    """Return the interpolated peak azimuth of one virtual-array snapshot."""
    spec, theta = angle_fft(
        snapshot, n_angle=n_angle, window=window, spacing=spacing, lam=lam
    )
    power = np.abs(spec) ** 2
    # Work in dB so a slightly flattened Hann mainlobe interpolates cleanly.
    mag_db = 10.0 * np.log10(power + 1e-30)
    peak = int(np.argmax(power))
    peak_f = _quadratic_bin(mag_db, peak)
    # Linear interpolation of the angle axis (uniform in sinθ, not θ).
    i0 = int(np.floor(peak_f))
    frac = peak_f - i0
    if 0 <= i0 < theta.size - 1:
        az = (1.0 - frac) * theta[i0] + frac * theta[i0 + 1]
    else:
        az = theta[peak]
    n_virt = np.asarray(snapshot).size
    return {
        "azimuth_rad": float(az),
        "azimuth_deg": float(np.degrees(az)),
        "peak_bin": peak,
        "peak_bin_interp": float(peak_f),
        "spectrum": spec,
        "theta": theta,
        "power": power,
        "resolution_rad": 2.0 / n_virt,
        "resolution_deg": float(np.degrees(2.0 / n_virt)),
    }


def angular_resolution(n_virt: int = N_VIRT) -> tuple[float, float]:
    """Broadside Rayleigh resolution 2/N rad (and deg) for a λ/2 ULA."""
    rad = 2.0 / n_virt
    return rad, float(np.degrees(rad))


def process_frame(cube: np.ndarray, *, mti: bool = False, compensate: bool = True) -> np.ndarray:
    """Convenience: IF cube -> Doppler-compensated range-Doppler-virtual cube."""
    rd = range_doppler_cube(cube, mti=mti)
    return compensate_tdm_doppler(rd) if compensate else rd


def _nearest_bin(axis: np.ndarray, value: float) -> int:
    return int(np.argmin(np.abs(axis - value)))


def _self_test() -> None:
    """A far-field tone at a known angle must land on that angle."""
    theta0 = np.deg2rad(20.0)
    n = np.arange(N_VIRT)
    snap = np.exp(1j * 2.0 * np.pi * SPACING * n * np.sin(theta0) / LAM)
    snap += 0.01 * (np.random.default_rng(0).standard_normal(N_VIRT)
                    + 1j * np.random.default_rng(1).standard_normal(N_VIRT))
    est = estimate_azimuth(snap, n_angle=256)
    err_deg = abs(est["azimuth_deg"] - 20.0)
    assert err_deg < 0.5, f"synthetic 20 deg recovered as {est['azimuth_deg']:.3f}"
    print(f"self-test: 20.00 deg tone recovered as {est['azimuth_deg']:+.2f} deg "
          f"(error {err_deg:.3f} deg)")


def _make_plot(
    ra_raw: np.ndarray,
    ra_mti: np.ndarray,
    theta: np.ndarray,
    r_axis: np.ndarray,
    det: dict,
    spectrum_info: dict,
    out_path: Path,
) -> None:
    import matplotlib.pyplot as plt

    def _db(power: np.ndarray) -> np.ndarray:
        db = 10.0 * np.log10(power + 1e-30)
        return db - np.nanmax(db)

    # Angle-FFT bins are uniform in sin(theta), not in degrees; plot against
    # the true azimuth so the person at ~+18 deg sits on the correct tick.
    th_deg = np.degrees(theta)
    rr = np.concatenate([r_axis, r_axis[-1:] + (r_axis[1] - r_axis[0])])
    # pcolormesh wants bin edges; mid-point theta is close enough after a
    # half-bin pad so cells centre on the true angles.
    dsin = np.sin(theta[1]) - np.sin(theta[0])
    sin_e = np.sin(theta[0]) - 0.5 * dsin + np.arange(theta.size + 1) * dsin
    th_edges = np.degrees(np.arcsin(np.clip(sin_e, -1.0, 1.0)))

    fig, axes = plt.subplots(1, 3, figsize=(13.5, 5.6), constrained_layout=True)
    fig.suptitle("RaDICaL frame 30 — range-angle maps and the CFAR cell")

    for ax, power, title in (
        (axes[0], ra_raw, "Raw (max over Doppler)"),
        (axes[1], ra_mti, "MTI (max over Doppler)"),
    ):
        im = ax.pcolormesh(
            th_edges,
            rr,
            _db(power),
            shading="auto",
            vmin=-35,
            vmax=0,
            cmap="viridis",
        )
        ax.set_xlabel("Azimuth (deg)")
        ax.set_ylabel("Range (m)")
        ax.set_ylim(0, 12)
        ax.set_xlim(-90, 90)
        ax.set_title(title)
        fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="dB, peak-normalized")

    ax = axes[1]
    ax.scatter(
        [det["azimuth_deg"]],
        [det["range_m"]],
        s=90,
        facecolors="none",
        edgecolors="white",
        linewidths=1.6,
        zorder=5,
    )
    ax.plot(det["azimuth_deg"], det["range_m"], "wx", ms=7, mew=1.4, zorder=6)
    ax.annotate(
        f"{det['range_m']:.2f} m\n{det['azimuth_deg']:+.1f} deg",
        (det["azimuth_deg"], det["range_m"]),
        textcoords="offset points",
        xytext=(10, 12),
        color="white",
        fontsize=8,
    )

    ax = axes[2]
    th_deg = np.degrees(spectrum_info["theta"])
    spec_db = 10.0 * np.log10(spectrum_info["power"] + 1e-30)
    spec_db -= spec_db.max()
    ax.plot(th_deg, spec_db, color="C0", lw=1.4)
    ax.axvline(det["azimuth_deg"], color="0.35", ls="--", lw=1.0)
    ax.plot(det["azimuth_deg"], 0.0, "kx", ms=8, mew=1.6)
    res_deg = det["resolution_deg"]
    ax.axvspan(
        det["azimuth_deg"] - res_deg / 2.0,
        det["azimuth_deg"] + res_deg / 2.0,
        color="C0",
        alpha=0.12,
        label=f"±½ (2/N) = ±{res_deg / 2:.1f} deg",
    )
    ax.set_xlim(-90, 90)
    ax.set_ylim(-35, 3)
    ax.set_xlabel("Azimuth (deg)")
    ax.set_ylabel("Power (dB, peak-normalized)")
    ax.set_title(
        f"Angle FFT at {det['range_m']:.2f} m, {det['velocity_mps']:+.2f} m/s"
    )
    ax.legend(loc="lower center", fontsize=8, frameon=False)
    ax.grid(True, alpha=0.3)

    fig.savefig(out_path, dpi=140)
    plt.close(fig)


def main(argv: list[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description=__doc__.split("\n\n")[0])
    parser.add_argument(
        "--h5",
        type=Path,
        default=Path("/workspace/radar_recordings/indoor_sample_50.h5"),
    )
    parser.add_argument("--frame", type=int, default=30)
    parser.add_argument("--range-m", type=float, default=2.53)
    parser.add_argument("--vel-mps", type=float, default=-0.62)
    parser.add_argument("--n-angle", type=int, default=128)
    parser.add_argument(
        "--plot",
        type=Path,
        default=Path("/workspace/radical_range_angle_map.png"),
    )
    args = parser.parse_args(argv)

    _self_test()

    import h5py

    with h5py.File(args.h5, "r") as f:
        cube = np.asarray(f["radar"][args.frame])
    print(f"loaded {args.h5} frame {args.frame}: {cube.shape} {cube.dtype}")

    r_axis = range_axis(cube.shape[-1])
    v_axis = vel_axis(cube.shape[0])
    ir = _nearest_bin(r_axis, args.range_m)
    iv = _nearest_bin(v_axis, args.vel_mps)
    print(
        f"CFAR cell: requested {args.range_m:.2f} m, {args.vel_mps:+.2f} m/s "
        f"-> bin ({ir}, {iv}) = {r_axis[ir]:.2f} m, {v_axis[iv]:+.2f} m/s"
    )

    rd_raw = range_doppler_cube(cube, mti=False)
    rd_mti = range_doppler_cube(cube, mti=True)
    rd_raw_c = compensate_tdm_doppler(rd_raw)
    rd_mti_c = compensate_tdm_doppler(rd_mti)

    snap = rd_mti_c[iv, :, ir]
    snap_unc = rd_mti[iv, :, ir]
    est = estimate_azimuth(snap, n_angle=args.n_angle)
    est_unc = estimate_azimuth(snap_unc, n_angle=args.n_angle)
    res_rad, res_deg = angular_resolution(cube.shape[1])

    # Nearby range bins on the same Doppler column, to see if the peak is stable.
    nearby = []
    for j in range(ir - 4, ir + 8):
        if 0 <= j < r_axis.size:
            e = estimate_azimuth(rd_mti_c[iv, :, j], n_angle=args.n_angle)
            nearby.append((j, r_axis[j], e["azimuth_deg"]))

    ra_raw, theta = range_angle_map(rd_raw, n_angle=args.n_angle, compensate=True)
    ra_mti, _ = range_angle_map(rd_mti, n_angle=args.n_angle, compensate=True)

    # Peak of the MTI range-angle map near the person's range (sanity check).
    r_lo, r_hi = ir - 6, ir + 12
    local = ra_mti[r_lo:r_hi, :]
    loc = np.unravel_index(int(np.argmax(local)), local.shape)
    ra_peak_r = r_axis[r_lo + loc[0]]
    ra_peak_az = float(np.degrees(theta[loc[1]]))

    det = {
        "range_m": float(r_axis[ir]),
        "velocity_mps": float(v_axis[iv]),
        "range_bin": ir,
        "doppler_bin": iv,
        "azimuth_rad": est["azimuth_rad"],
        "azimuth_deg": est["azimuth_deg"],
        "azimuth_deg_uncompensated": est_unc["azimuth_deg"],
        "resolution_rad": res_rad,
        "resolution_deg": res_deg,
    }

    print(
        f"array: {N_VIRT}-element TDM-MIMO virtual ULA, "
        f"d = λ/2 = {SPACING * 1e3:.3f} mm at {FC/1e9:.0f} GHz"
    )
    print(
        f"angular resolution 2/N = {res_rad:.4f} rad = {res_deg:.2f} deg "
        f"(config header quotes 15 deg)"
    )
    print(
        f"azimuth at CFAR cell (TDM-compensated): "
        f"{est['azimuth_deg']:+.2f} deg ({est['azimuth_rad']:+.4f} rad)"
    )
    print(
        f"azimuth without TDM compensation: "
        f"{est_unc['azimuth_deg']:+.2f} deg "
        f"(delta {est['azimuth_deg'] - est_unc['azimuth_deg']:+.2f} deg)"
    )
    print(
        f"MTI range-angle peak near 2.5-3.5 m: "
        f"{ra_peak_r:.2f} m, {ra_peak_az:+.1f} deg"
    )
    print("nearby range bins at the same Doppler (compensated):")
    for j, r, az in nearby:
        mark = " <-- CFAR" if j == ir else ""
        print(f"  bin {j:3d}  {r:5.2f} m  {az:+6.1f} deg{mark}")

    _make_plot(ra_raw, ra_mti, theta, r_axis, det, est, args.plot)
    print(f"saved {args.plot} ({args.plot.stat().st_size} bytes)")

    # Compact summary for the Page heatmap (max-pool to 16 x 16).
    def _pool(power: np.ndarray, n_r: int = 16, n_a: int = 16) -> np.ndarray:
        db = 10.0 * np.log10(power + 1e-30)
        db = db - db.max()
        # restrict display to 0-12 m, ±90 deg (already the full axis)
        r_mask = r_axis <= 12.0
        sl = db[r_mask]
        rs = sl.shape[0] // n_r * n_r
        as_ = sl.shape[1] // n_a * n_a
        sl = sl[:rs, :as_]
        sl = sl.reshape(n_r, rs // n_r, n_a, as_ // n_a).max(axis=(1, 3))
        sl = np.clip(sl + 35.0, 0.0, 35.0)  # 0 = floor at -35 dB
        return sl

    np.set_printoptions(precision=1, linewidth=120, suppress=True)
    print("MTI RA pooled (0-12 m x ±90 deg, 0..35 = dB above -35):")
    print(_pool(ra_mti))
    print("RAW RA pooled:")
    print(_pool(ra_raw))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())
'''
open(os.path.join(HELPERS, 'angle_fft.py'), 'w').write(SRC)
print('wrote angle_fft.py', len(SRC), 'chars')

In [ ]:
import fft2d_explainer as fx
import windowing_compare as wc
import ca_cfar_2d as cf
import angle_fft as af
print("helpers imported")

## Module 1 · FFT basics
Goal: see what an FFT bin means, why off-bin tones leak, and how a Hann window and zero padding change the picture.

Done when: you can predict which bin a tone lands in and explain the leakage you see.

In [ ]:
fs, N = 1000, 64                 # 1 kHz sampling, 64 samples -> 15.6 Hz bins
t = np.arange(N) / fs
x = np.exp(2j*np.pi*125*t)                              # exactly 8 cycles
y = np.exp(2j*np.pi*130*t)                              # 8.32 cycles: leaks
f = np.fft.fftshift(np.fft.fftfreq(N, 1/fs))
for sig, lab in [(x, "125 Hz (on-bin)"), (y, "130 Hz (off-bin)")]:
    S = np.fft.fftshift(np.fft.fft(sig))
    plt.plot(f, 20*np.log10(np.abs(S)+1e-9), ".-", label=lab)
plt.ylim(-40, 40); plt.xlabel("Hz"); plt.ylabel("dB"); plt.legend()
plt.title("Bin spacing fs/N = %.1f Hz" % (fs/N)); plt.show()

In [ ]:
# A naive DFT, equation (10) on the Page, matches np.fft.fft
def naive_dft(x):
    n = np.arange(len(x))
    return np.exp(-2j*np.pi*np.outer(n, n)/len(x)) @ x

v = np.random.randn(64) + 1j*np.random.randn(64)
print("max |DFT - FFT| =", np.abs(naive_dft(v) - np.fft.fft(v)).max())

In [ ]:
# Hann window + 8x zero padding on the leaking 130 Hz tone
w = hann(N, sym=False)
for sig, lab, n_fft in [(y, "rect", N), (y*w, "Hann", N), (y*w, "Hann, 8x pad", 8*N)]:
    S = np.fft.fftshift(np.fft.fft(sig, n=n_fft))
    fk = np.fft.fftshift(np.fft.fftfreq(n_fft, 1/fs))
    plt.plot(fk, 20*np.log10(np.abs(S)+1e-9), label=lab)
    print(f"{lab:13s} peak at {fk[np.argmax(np.abs(S))]:6.1f} Hz")
plt.xlim(-100, 400); plt.ylim(-60, 40); plt.legend(); plt.xlabel("Hz"); plt.show()

## Module 2 · Range-Doppler on a simulated cube
Three point targets: 50 m at 0 m/s, 80 m at +8 m/s, 120 m at −12 m/s. 77 GHz, 150 MHz sweep in 40 µs, 256 samples × 128 chirps.

Done when: three peaks show at the right range and velocity.

In [ ]:
cube = fx.make_sim_cube()          # (256 samples, 128 chirps)
NS, NC, T, LAM = fx.NS, fx.NC, fx.T, fx.LAM
dR = fx.C / (2*fx.B)
r_axis = np.arange(NS) * dR
v_axis = (np.arange(NC) - NC//2) * LAM / (2*NC*T)
print(cube.shape, "range res", dR, "m | vel res", round(LAM/(2*NC*T), 3), "m/s")

In [ ]:
# Range FFT along fast time (axis 0)
rng = np.fft.fft(cube * hann(NS, sym=False)[:, None], axis=0)
prof = 20*np.log10(np.abs(rng).mean(axis=1) + 1e-9)
plt.plot(r_axis[:NS//2], prof[:NS//2]); plt.xlabel("range (m)"); plt.ylabel("dB")
plt.title("Range profile (averaged over chirps)"); plt.show()

In [ ]:
# Doppler FFT along slow time (axis 1), then fftshift
rd = np.fft.fftshift(np.fft.fft(rng * hann(NC, sym=False)[None, :], axis=1), axes=1)
rd_db = 20*np.log10(np.abs(rd[:NS//2]) + 1e-9)
plt.pcolormesh(v_axis, r_axis[:NS//2], rd_db, vmin=rd_db.max()-50, shading="auto")
plt.xlabel("velocity (m/s)"); plt.ylabel("range (m)"); plt.colorbar(label="dB")
plt.title("Simulated range-Doppler map"); plt.show()

## Module 2 (cont.) · 2D FFT, explained
Range FFT then Doppler FFT is exactly `np.fft.fft2`; the two axes commute. Cost: N·M·log(N·M).

In [ ]:
stats = fx.compare_to_fft2(cube)
for k, v in stats.items():
    if np.isscalar(v): print(f"{k:32s} {v:.3e}")

In [ ]:
imgs = fx.make_example_images()
fig, axs = plt.subplots(2, len(imgs), figsize=(7.6, 4.0))
for j, (name, img) in enumerate(imgs.items()):
    axs[0, j].imshow(img, cmap="gray"); axs[0, j].set_title(name, fontsize=8)
    axs[1, j].imshow(fx.spectrum_db(img), cmap="magma", vmin=-60, vmax=0)
for a in axs[1]: a.set_xlim(40, 88); a.set_ylim(88, 40)   # zoom on the centre
for a in axs.flat: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## Module 3 · Windowing
Rectangular, Hann, Blackman, Chebyshev 80 dB: peak sidelobe level (PSL), −3 dB width, processing loss. Then a −40 dB target 3 bins from a strong one.

Done when: you can say which window you would pick for weak targets near strong ones.

In [ ]:
print(f"{'window':16s} {'PSL dB':>7s} {'-3dB bins':>9s} {'loss dB':>8s}")
for name in wc.WINDOW_NAMES:
    m = wc.window_metrics(wc.make_window(name, 256))
    print(f"{name:16s} {m['psl_db']:7.1f} {m['width_3db_bins']:9.2f} {m['processing_loss_db']:8.2f}")

In [ ]:
# Masking demo: strong target slightly off-grid, -40 dB target 3 bins away
mask_cube = wc.make_sim_cube(targets=[(60.4, 0.0, 1.0), (63.4, 0.0, 0.01)], noise_std=0.0)
for name in ["rectangular", "blackman"]:
    r, p = wc.range_cut_sim(mask_cube, wc.make_window(name, wc.NS), wc.make_window(name, wc.NC))
    db = 10*np.log10(p + 1e-30); plt.plot(r, db - db.max(), label=name)
plt.axvline(63.4, ls="--", c="gray"); plt.xlim(52, 72); plt.ylim(-90, 3)
plt.xlabel("range (m)"); plt.ylabel("dB"); plt.legend(); plt.title("-40 dB target at 63.4 m"); plt.show()

## Module 4 · Real data: RaDICaL
`indoor_sample_50.h5` is 279 MB (50 frames of a TI xWR1443 at 77 GHz, a person walking indoors, plus camera and depth images). This app bundles only the radar cubes of frames 0, 10, 20, 25, 30, 35, 40, 45 and 49 in `radical_frames.npz` (2.2 MB). The samples are integers, so they are stored as int16 I/Q with no loss: every number below is the same as from the full file.

In [ ]:
#@title Load the bundled RaDICaL frames
NPZ = "radical_frames.npz"
if os.path.exists(NPZ):
    data = np.load(NPZ)
else:                                     # browser fallback: fetch it from the site
    import js
    from pyodide.http import pyfetch
    root = str(js.location.href).split("/extensions/")[0].split("/build/")[0]
    resp = await pyfetch(root + "/files/" + NPZ)
    data = np.load(io.BytesIO(await resp.bytes()))
FRAME_IDS = [int(k) for k in data["frame_ids"]]
IQ = data["radar_iq"]                     # (frames, 32 chirps, 8 virtual RX, 304 samples, I/Q)

def get_frame(k):
    """Complex cube (32 chirps, 8 virtual RX, 304 samples) for RaDICaL frame k."""
    i = FRAME_IDS.index(k)
    return IQ[i, ..., 0].astype(np.float64) + 1j * IQ[i, ..., 1]

print("frames", FRAME_IDS, "|", IQ.shape, IQ.dtype)

In [ ]:
frame = get_frame(30)                     # (32 chirps, 8 virtual RX, 304 samples)
r_ax, v_ax = af.range_axis(), af.vel_axis()
print(frame.shape, frame.dtype, "| range res", round(af.RANGE_RES, 4), "m | vel res", round(af.VEL_RES, 3), "m/s")

In [ ]:
def rd_power(cube, mti=False):
    rd = af.range_doppler_cube(cube, mti=mti)       # (doppler, virt, range)
    return np.mean(np.abs(rd)**2, axis=1).T         # (range, doppler)

def show_rd(p, title, rmax=6):
    db = 10*np.log10(p + 1e-12); k = r_ax <= rmax
    plt.pcolormesh(v_ax, r_ax[k], db[k], vmin=db.max()-45, shading="auto")
    plt.xlabel("velocity (m/s)"); plt.ylabel("range (m)"); plt.title(title); plt.colorbar(label="dB")

raw = rd_power(frame)
show_rd(raw, "RaDICaL frame 30, raw"); plt.show()

## Module 5 · MTI (static clutter removal)
Subtract each channel's slow-time mean before the Doppler FFT. Walls at 0 m/s vanish; the walking person stays.

In [ ]:
mti = rd_power(frame, mti=True)
show_rd(mti, "Frame 30 after MTI"); plt.show()

## Module 6 · CA-CFAR detection
Threshold = α × (mean of training cells), α = N(P_fa^(−1/N) − 1). Run on the synthetic cube (expect 3 targets) and on the MTI map (expect the person near 2.5 m, −0.6 m/s).

In [ ]:
sim_pow = np.abs(rd)**2                                   # (range, doppler) from Module 2
dets_sim, _ = cf.detect(sim_pow, r_axis, v_axis, guard=(2, 2), train=(6, 6), pfa=1e-4,
                        min_range=1.0, min_snr_db=15, merge_range=2.0, merge_vel=0.8)
for d in dets_sim:
    print(f"sim: {d['range_m']:6.1f} m  {d['velocity_mps']:+6.2f} m/s  SNR {d['snr_db']:.1f} dB")

In [ ]:
CFAR = dict(guard=(3, 1), train=(10, 4), pfa=1e-5, min_range=0.4,
            min_abs_vel=0.4, min_snr_db=14, merge_range=1.2, merge_vel=0.7)
dets, res = cf.detect(mti, r_ax, v_ax, **CFAR)
for d in dets:
    print(f"real: {d['range_m']:5.2f} m  {d['velocity_mps']:+5.2f} m/s  SNR {d['snr_db']:.1f} dB")
show_rd(mti, "CA-CFAR on frame 30 (MTI)")
for d in dets: plt.plot(d["velocity_mps"], d["range_m"], "wo", mfc="none", ms=12, mew=2)
plt.show()

In [ ]:
# Sanity check across a few frames
for k in (0, 10, 25, 30, 40):
    dd, _ = cf.detect(rd_power(get_frame(k), mti=True), r_ax, v_ax, **CFAR)
    print(k, [(round(d["range_m"], 2), round(d["velocity_mps"], 2)) for d in dd])

## Module 7 · Angle FFT
TDM-MIMO: 2 TX × 4 RX = 8-element virtual ULA at λ/2. Compensate the Doppler phase from TX switching, then FFT across the 8 channels. Resolution ≈ 2/N rad ≈ 14° at broadside.

In [ ]:
rd_m = af.process_frame(frame, mti=True)                  # compensated (doppler, virt, range)
d0 = dets[0]
snap = rd_m[d0["doppler_bin"], :, d0["range_bin"]]
az = af.estimate_azimuth(snap)
print(f"person at {d0['range_m']:.2f} m, {d0['velocity_mps']:+.2f} m/s -> azimuth {az['azimuth_deg']:+.1f} deg"
      f" (resolution ~{az['resolution_deg']:.1f} deg)")

In [ ]:
pw, th = af.range_angle_map(af.range_doppler_cube(frame, mti=True))
k = r_ax <= 6
db = 10*np.log10(pw[k] + 1e-12)
plt.pcolormesh(np.degrees(th), r_ax[k], db, vmin=db.max()-30, shading="auto")
plt.plot(az["azimuth_deg"], d0["range_m"], "wo", mfc="none", ms=12, mew=2)
plt.xlabel("azimuth (deg)"); plt.ylabel("range (m)"); plt.title("Range-angle map, MTI"); plt.colorbar(label="dB")
plt.show()

## Optional next steps
- Tracking: run Modules 5–7 on the 9 bundled frames (`FRAME_IDS`) and plot range vs time.
- Micro-Doppler: a short-time FFT across consecutive frames at the person's range bin.
- Your own hardware: swap in your radar's config constants in `angle_fft.py`.